In [ ]:
!pip install peft accelerate bitsandbytes av pillow tensorboard

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    LlavaNextVideoProcessor,
    LlavaNextVideoForConditionalGeneration,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import av
import cv2
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Optional
import json
import os
from PIL import Image
from google.colab import drive

In [ ]:
# Mount to specific google drive path
base_path = '/content/drive/My Drive/Colab Notebooks/PhD Thesis Code Notebooks/datasets'
drive.mount('/content/drive')
os.chdir(base_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [ ]:
# Custom Dataset for Video QA
class VideoQADataset(Dataset):
    """
    Dataset for video question-answering tasks.
    Expected data format in JSON:
    [
        {
            "video_path": "path/to/video.mp4",
            "conversations": [
                {"role": "user", "content": "What is happening in the video?"},
                {"role": "assistant", "content": "A person is walking..."}
            ]
        }
    ]
    """
    def __init__(self, data_path: str, processor, max_frames: int = 8):
        with open(data_path, 'r') as f:
            self.data = json.load(f)
        self.processor = processor
        self.max_frames = max_frames

    def __len__(self):
        return len(self.data)

    def read_video_pyav(self, video_path: str):
        """Read video frames using PyAV with error handling"""
        try:
            container = av.open(video_path)

            # Get video stream
            video_stream = container.streams.video[0]

            # Calculate frame indices to sample
            total_frames = video_stream.frames
            if total_frames == 0 or total_frames is None:
                # Fallback: decode all frames and count
                frames_list = []
                for frame in container.decode(video=0):
                    frames_list.append(frame.to_ndarray(format="rgb24"))
                total_frames = len(frames_list)

                # Sample frames uniformly
                indices = np.linspace(0, total_frames - 1, min(self.max_frames, total_frames), dtype=int)
                sampled_frames = [Image.fromarray(frames_list[i]) for i in indices]
                container.close()
                return sampled_frames

            # Normal path: sample frames by index
            indices = np.linspace(0, total_frames - 1, min(self.max_frames, total_frames), dtype=int)

            frames = []
            container.seek(0)
            for i, frame in enumerate(container.decode(video=0)):
                if i in indices:
                    frames.append(frame.to_ndarray(format="rgb24"))
                if len(frames) >= self.max_frames:
                    break

            container.close()

            # Ensure we have frames
            if len(frames) == 0:
                raise ValueError(f"No frames extracted from video: {video_path}")

            return [Image.fromarray(f) for f in frames]

        except Exception as e:
            print(f"Error reading video {video_path}: {str(e)}")
            print(f"Skipping corrupted video file.")
            # Return None to signal this video should be skipped
            return None

      # Function to read video frames using OpenCV
    def read_video_opencv(self, video_path):
      '''
      Decode video frames using OpenCV.
      Args:
          video_path (str): Path to the video file.
          num_frames (int): Number of frames to decode.
      Returns:
          result (np.ndarray): Array of decoded frames of shape (num_frames, height, width, 3).
      '''
      try:
        frames = []
        index = 0
        video = cv2.VideoCapture(video_path)
        total_num_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.arange(0, total_num_frames, total_num_frames / self.max_frames).astype(int)
        while video.isOpened():
            success, frame = video.read()
            if index in indices:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            if success:
                index += 1
            if index >= total_num_frames:
                break

        video.release()

        if not frames:
            return None

        return [Image.fromarray(f) for f in frames]
      except Exception as e:
          print(f"Error reading video {video_path}: {str(e)}")
          return None

    def __getitem__(self, idx):
        item = self.data[idx]
        video_path = item['video_path']
        conversations = item['conversations']

        # Read video frames
        frames = self.read_video_opencv(video_path)

        # Skip corrupted videos
        if frames is None:
            # Return next valid item
            return self.__getitem__((idx + 1) % len(self.data))

        # Format conversation
        prompt = conversations[0]['content']
        answer = conversations[1]['content']

        # Create the full conversation text
        conversation = f"USER: <video>\n{prompt}\nASSISTANT: {answer}"

        return {
            'video': frames,
            'text': conversation,
            'answer': answer
        }

In [ ]:
@dataclass
class DataCollator:
    """Custom data collator for video-text pairs"""
    processor: LlavaNextVideoProcessor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        videos = [f['video'] for f in features]
        texts = [f['text'] for f in features]

        # Process inputs
        batch = self.processor(
            text=texts,
            videos=videos,
            padding=True,
            return_tensors="pt"
        )

        # Create labels (same as input_ids for causal LM)
        labels = batch['input_ids'].clone()

        # Mask padding tokens in labels
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        # Optionally: mask the prompt part to only compute loss on the answer
        # This requires finding where "ASSISTANT:" starts in each sequence

        batch['labels'] = labels
        return batch

In [ ]:
def setup_model_and_processor(model_name: str = "llava-hf/LLaVA-NeXT-Video-7B-hf"):
    """Load model and processor with LoRA configuration"""

    # Quantization config for memory efficiency
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    # Load processor
    processor = LlavaNextVideoProcessor.from_pretrained(model_name)

    # Load model
    model = LlavaNextVideoForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )

    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model)

    # LoRA configuration
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Add LoRA adapters
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model, processor

In [ ]:
def train_model(model,
                processor,
                train_dataset,
                eval_dataset=None,
                output_dir: str = "./llava_video_finetuned",
                num_epochs: int = 3,
                batch_size: int = 1,
                gradient_accumulation_steps: int = 8,
                learning_rate: float = 2e-4,
            ):
    """Fine-tune the model"""

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        weight_decay=0.05,
        warmup_ratio=0.1,
        logging_steps=10,
        max_grad_norm=0.5,
        save_steps=100,
        eval_steps=20 if eval_dataset else None,
        eval_strategy="steps" if eval_dataset else "no",
        save_total_limit=2,
        fp16=True,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
        report_to="tensorboard",
        load_best_model_at_end=True if eval_dataset else False,
    )

    # Data collator
    data_collator = DataCollator(processor=processor)

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )

    # Train
    print("Starting training...")
    trainer.train()

    # Save final model
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)
    print(f"Model saved to {output_dir}")

    return trainer

In [ ]:
def save_and_push_model(model,
                      processor,
                      output_dir: str,
                      hub_model_id: Optional[str] = None,
                      merge_adapters: bool = True
                  ):
    """
    Save model and optionally push to Hugging Face Hub

    Args:
        model: Trained model with LoRA adapters
        processor: Model processor
        output_dir: Local directory to save model
        hub_model_id: Hugging Face Hub model ID (e.g., "username/model-name")
        merge_adapters: If True, merge LoRA adapters into base model for standalone loading
    """

    if merge_adapters:
        print("\nMerging LoRA adapters with base model...")
        # Merge adapters into the base model
        model = model.merge_and_unload()

        # Save merged model
        merged_dir = output_dir + "_merged"
        print(f"Saving merged model to {merged_dir}...")
        model.save_pretrained(merged_dir)
        processor.save_pretrained(merged_dir)
        print(f"Merged model saved to {merged_dir}")

        # Push to Hub if specified
        if hub_model_id:
            print(f"\nPushing merged model to Hugging Face Hub: {hub_model_id}")
            model.push_to_hub(hub_model_id)
            processor.push_to_hub(hub_model_id)
            print(f"✓ Merged model pushed to: https://huggingface.co/{hub_model_id}")
            print(f"\nTo load this model:")
            print(f"  model = LlavaNextVideoForConditionalGeneration.from_pretrained('{hub_model_id}')")
            print(f"  processor = LlavaNextVideoProcessor.from_pretrained('{hub_model_id}')")

    else:
        # Save LoRA adapters only (smaller size)
        print(f"\nSaving LoRA adapters to {output_dir}...")
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"LoRA adapters saved to {output_dir}")

        # Push to Hub if specified
        if hub_model_id:
            print(f"\nPushing LoRA adapters to Hugging Face Hub: {hub_model_id}")
            model.push_to_hub(hub_model_id)
            processor.push_to_hub(hub_model_id)
            print(f"✓ LoRA adapters pushed to: https://huggingface.co/{hub_model_id}")
            print(f"\nTo load this model:")
            print(f"  base_model = LlavaNextVideoForConditionalGeneration.from_pretrained('llava-hf/LLaVA-NeXT-Video-7B-hf')")
            print(f"  model = PeftModel.from_pretrained(base_model, '{hub_model_id}')")
            print(f"  processor = LlavaNextVideoProcessor.from_pretrained('llava-hf/LLaVA-NeXT-Video-7B-hf')")

In [ ]:
# Main execution
# Configuration
MODEL_NAME = "llava-hf/LLaVA-NeXT-Video-7B-hf"
TRAIN_DATA_PATH = "train_dataset.json"
EVAL_DATA_PATH = "val_dataset.json"
OUTPUT_DIR = "./llava_video_finetuned"
MAX_FRAMES = 8
NUM_EPOCHS = 5
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-3

# Setup
print("Loading model and processor...")
model, processor = setup_model_and_processor(MODEL_NAME)

# Load datasets
print("Loading datasets...")
train_dataset = VideoQADataset(TRAIN_DATA_PATH, processor, max_frames=MAX_FRAMES)
eval_dataset = VideoQADataset(EVAL_DATA_PATH, processor, max_frames=MAX_FRAMES) if os.path.exists(EVAL_DATA_PATH) else None

# Train
trainer = train_model(
    model=model,
    processor=processor,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE
)

print("Training complete!")

Loading model and processor...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

trainable params: 19,136,512 || all params: 7,082,567,680 || trainable%: 0.2702
Loading datasets...
Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss
20,10.461700,5.287175
40,3.993400,3.850942
60,3.728000,3.729169
80,3.727100,3.743718
100,3.710500,3.714538
120,3.690800,3.702077
140,3.684200,3.697108
160,3.690000,3.693774
180,3.686600,3.689173
200,3.674000,3.686404


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default

Model saved to ./llava_video_finetuned
Training complete!


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
hub_model_id = "saravit-soeng/llava-next-video-7b-hf-abnormal-action-fine-tuned-v4"
save_and_push_model(trainer.model, processor, trainer.args.output_dir, hub_model_id)


Merging LoRA adapters with base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Saving merged model to ./llava_video_finetuned_merged...
Merged model saved to ./llava_video_finetuned_merged

Pushing merged model to Hugging Face Hub: saravit-soeng/llava-next-video-7b-hf-abnormal-action-fine-tuned-v4


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p2ye32mzs/tokenizer.model: 100%|##########|  500kB /  500kB            

✓ Merged model pushed to: https://huggingface.co/saravit-soeng/llava-next-video-7b-hf-abnormal-action-fine-tuned-v4

To load this model:
  model = LlavaNextVideoForConditionalGeneration.from_pretrained('saravit-soeng/llava-next-video-7b-hf-abnormal-action-fine-tuned-v4')
  processor = LlavaNextVideoProcessor.from_pretrained('saravit-soeng/llava-next-video-7b-hf-abnormal-action-fine-tuned-v4')
